<div style="font-size:2em; font-weight:bold; margin-bottom:8px;">02 — Clean &amp; Normalize the Corpus Text</div>

This notebook reads the raw SciFact corpus produced by notebook 01 and cleans the text so it is ready for reliable chunking and embedding.

It is **Step 2** of the RAG data indexing pipeline — text cleaning only.

---

**What this notebook does:**
1. Loads the normalized corpus from `data/raw/corpus/corpus.jsonl`
2. Explores the text for common quality problems
3. Defines small, testable cleaning helper functions
4. Tests the cleaning on a single document so you can see before/after
5. Cleans every document and drops the ones with no usable content
6. Saves the result to `data/processed/02_clean_corpus.jsonl`
7. Reads the output back to verify it looks correct

**What this notebook intentionally does NOT do:**
- No chunking
- No metadata enrichment
- No embedding

> **Before running:** make sure dependencies are installed.
> ```bash
> pip install -r requirements.txt
> ```
> The cleaning step uses `ftfy` to repair broken or garbled unicode.

---
## 1. Imports

We need a small set of tools:
- **`json`** — read and write JSON Lines files
- **`re`** — regular expressions for whitespace and control-character cleanup
- **`unicodedata`** — normalize unicode to a consistent canonical form (NFC)
- **`pathlib.Path`** — OS-independent file paths
- **`ftfy`** — "fixes text for you": repairs broken unicode / mojibake (e.g. `â€™` → `’`)

In [1]:
# ── [1 / 10] Imports ────────────────────────────────────────────────────────

import json
import re
import unicodedata
from pathlib import Path

import ftfy

print("Imports ready.")

Imports ready.


---
## 2. Configuration — Input and Output Paths

All paths live in one place so they are easy to change.

The same project-root detection used in notebook 01 is reused here, so the
notebook works whether it is launched from `notebooks/` or from the project root.

In [2]:
# ── [2 / 10] Configuration ──────────────────────────────────────────────────

# Resolve the project root no matter where the notebook is run from
_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

# Input — produced by notebook 01
INPUT_FILE = ROOT / "data" / "raw" / "corpus" / "corpus.jsonl"

# Output — cleaned corpus for the chunking step
OUTPUT_DIR  = ROOT / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "02_clean_corpus.jsonl"

print(f"Project root : {ROOT}")
print(f"Input file   : {INPUT_FILE}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"Input exists : {INPUT_FILE.exists()}")

Project root : /app
Input file   : /app/data/raw/corpus/corpus.jsonl
Output file  : /app/data/processed/02_clean_corpus.jsonl
Input exists : True


---
## 3. Load the Raw Corpus

We read the JSONL file produced by notebook 01 into a list of dictionaries.
Each line is one document with the schema: `document_id`, `title`, `text`, `source`, `dataset_config`.

If the input file is missing, we stop with a clear error telling you to run notebook 01 first.

In [3]:
# ── [3 / 10] Load the Raw Corpus ────────────────────────────────────────────

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Run notebook 01_download_scifact_corpus.ipynb first."
    )

documents = []
with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line))

print(f"Loaded {len(documents):,} documents")
print()
print("Example document keys:", list(documents[0].keys()))

Loaded 5,183 documents

Example document keys: ['document_id', 'title', 'text', 'source', 'dataset_config']


---
## 4. Explore — Find Text Quality Problems

Before cleaning, let's measure how common each problem is across the corpus.
This proves the cleaning is actually needed and lets us confirm it worked later (Step 10).

We scan for:

| Problem | Why it hurts the pipeline |
|---|---|
| Leading / trailing whitespace | Adds noise and shifts chunk boundaries |
| Multiple spaces / tabs | Wastes tokens and distorts token counts |
| 3+ consecutive newlines | Creates empty, meaningless chunks |
| Control characters | Can corrupt embeddings and storage |
| Non-normalized unicode | Same character stored multiple ways → inconsistent matching |

In [4]:
# ── [4 / 10] Explore — Count Text Problems ──────────────────────────────────

# Compile the patterns once so the scan over thousands of rows stays fast.
re_multi_space = re.compile(r"[ \t]{2,}")           # two or more spaces/tabs in a row
re_multi_blank = re.compile(r"\n{3,}")              # three or more newlines in a row
re_control     = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")  # control chars (keep \t \n \r)


def has_edge_whitespace(s: str) -> bool:
    """True if the string has leading or trailing whitespace."""
    return s != s.strip()


counts = {
    "edge_whitespace" : 0,
    "multi_space"     : 0,
    "multi_blank"     : 0,
    "control_chars"   : 0,
    "non_nfc_unicode" : 0,
}

for doc in documents:
    text = doc.get("text", "") or ""
    if has_edge_whitespace(text):
        counts["edge_whitespace"] += 1
    if re_multi_space.search(text):
        counts["multi_space"] += 1
    if re_multi_blank.search(text):
        counts["multi_blank"] += 1
    if re_control.search(text):
        counts["control_chars"] += 1
    if text != unicodedata.normalize("NFC", text):
        counts["non_nfc_unicode"] += 1

print(f"Scanned {len(documents):,} documents")
print()
print("Problem counts across the corpus:")
for name, n in counts.items():
    print(f"  {name:<18}: {n:,}")

Scanned 5,183 documents

Problem counts across the corpus:
  edge_whitespace   : 0
  multi_space       : 65
  multi_blank       : 0
  control_chars     : 0
  non_nfc_unicode   : 1


---
## 5. Look at a Concrete Example

Counting is abstract. Let's find one document that contains non-ASCII characters
(for example the curly apostrophe `’`, an en-dash `–`, or the Greek letter `β`)
and print its raw text so we can see exactly what we are about to fix.

In [5]:
# ── [5 / 10] Explore — Show a Document With Unicode Artifacts ────────────────

# Characters that the cleaner will normalize: curly quotes, dashes, Greek letters,
# and the non-breaking space (\u00a0).
artifact_chars = ("’", "“", "”", "–", "—", "β", "\u00a0")

example = None
for doc in documents:
    text = doc.get("text", "") or ""
    if any(ch in text for ch in artifact_chars):
        example = doc
        break

if example is None:
    print("No obvious unicode artifacts found — using the first document instead.")
    example = documents[0]
else:
    snippet = example["text"]
    print(f"document_id : {example['document_id']}")
    print(f"snippet     : ...{snippet[:300]}...")
    print()
    # Show the distinct non-ASCII characters present in this document
    non_ascii = sorted({c for c in snippet if ord(c) > 127})
    print("Non-ASCII characters present:", non_ascii[:20])

document_id : 5836
snippet     : ...Myelodysplastic syndromes (MDS) are age-dependent stem cell malignancies that share biological features of activated adaptive immune response and ineffective hematopoiesis. Here we report that myeloid-derived suppressor cells (MDSC), which are classically linked to immunosuppression, inflammation, a...

Non-ASCII characters present: ['β', '–', '’']


---
## 6. Cleaning Helper Functions

We build the cleaner from small, single-purpose functions. Each one is easy to
read, test, and reuse in the service layer later (Phase 7).

| Function | Responsibility |
|---|---|
| `fix_unicode` | Repair broken unicode with `ftfy` and normalize to NFC |
| `remove_control_chars` | Strip non-printable control characters (keeps `\n`, `\t`, `\r`) |
| `normalize_whitespace` | Collapse repeated spaces/tabs and trim each line |
| `collapse_blank_lines` | Reduce 3+ blank lines down to a single blank line |
| `clean_text` | Run all steps in the correct order |

The goal is **light** cleaning — enough for reliable chunking and embedding,
not heavy NLP preprocessing.

In [6]:
# ── [6 / 10] Cleaning Helper Functions ──────────────────────────────────────

def fix_unicode(text: str) -> str:
    """Repair garbled unicode and normalize to a single canonical form (NFC)."""
    text = ftfy.fix_text(text)
    return unicodedata.normalize("NFC", text)


def remove_control_chars(text: str) -> str:
    """Remove non-printable control characters but keep newlines and tabs."""
    return re_control.sub("", text)


def normalize_whitespace(text: str) -> str:
    """Collapse runs of spaces/tabs and strip trailing space on each line."""
    # Turn any tab or repeated space into a single space
    text = re.sub(r"[ \t]+", " ", text)
    # Strip leading/trailing spaces on every individual line
    text = "\n".join(line.strip() for line in text.split("\n"))
    return text


def collapse_blank_lines(text: str) -> str:
    """Reduce 3 or more consecutive newlines down to a single blank line."""
    return re_multi_blank.sub("\n\n", text)


def clean_text(text: str) -> str:
    """Full cleaning pipeline for a single string.

    Order matters: fix unicode first, then strip control chars, then handle
    whitespace, then collapse blank lines, and finally trim the edges.
    """
    if not text:
        return ""
    text = fix_unicode(text)
    text = remove_control_chars(text)
    text = normalize_whitespace(text)
    text = collapse_blank_lines(text)
    return text.strip()


print("Cleaning functions defined.")

Cleaning functions defined.


---
## 7. Test the Cleaner on a Crafted Example

Always test on a controlled example before processing thousands of rows.
We build a deliberately messy string that contains every problem at once,
clean it, and assert that each problem is gone. The notebook will fail loudly
here if the cleaning logic ever regresses.

In [7]:
# ── [7 / 10] Test the Cleaner on a Crafted Example ──────────────────────────

# This string packs in: leading/trailing spaces, double spaces, tabs,
# 4 blank lines, a control char (\x07 = bell), and unicode artifacts.
messy = "  Hello   world\t\tthis  is\n\n\n\nspaced   out.\x07  And CD33\u2019s role in TGF-\u03b2.  "

print("BEFORE (repr):")
print(repr(messy))
print()

cleaned = clean_text(messy)
print("AFTER (repr):")
print(repr(cleaned))
print()

# Assertions — the notebook stops here if any problem is left behind
assert "\t" not in cleaned,        "tabs should be gone"
assert "  " not in cleaned,        "double spaces should be gone"
assert "\n\n\n" not in cleaned,    "3+ newlines should be collapsed"
assert "\x07" not in cleaned,      "control chars should be removed"
assert cleaned == cleaned.strip(), "edges should be trimmed"
print("All cleaning assertions passed.")

BEFORE (repr):
'  Hello   world\t\tthis  is\n\n\n\nspaced   out.\x07  And CD33’s role in TGF-β.  '

AFTER (repr):
"Hello world this is\n\nspaced out. And CD33's role in TGF-β."

All cleaning assertions passed.


---
## 8. Clean a Real Document (Before / After)

Now run the cleaner on the real example document we found in Step 5 and compare
the character counts so you can see the effect on actual data.

In [8]:
# ── [8 / 10] Clean a Real Document ──────────────────────────────────────────

before_title = example.get("title", "") or ""
before_text  = example.get("text", "") or ""

after_title = clean_text(before_title)
after_text  = clean_text(before_text)

print(f"document_id                : {example['document_id']}")
print(f"title chars  before/after  : {len(before_title)} / {len(after_title)}")
print(f"text  chars  before/after  : {len(before_text)} / {len(after_text)}")
print()
print("Cleaned text preview:")
print(after_text[:300], "...")

document_id                : 5836
title chars  before/after  : 64 / 64
text  chars  before/after  : 1589 / 1589

Cleaned text preview:
Myelodysplastic syndromes (MDS) are age-dependent stem cell malignancies that share biological features of activated adaptive immune response and ineffective hematopoiesis. Here we report that myeloid-derived suppressor cells (MDSC), which are classically linked to immunosuppression, inflammation, a ...


---
## 9. Clean the Whole Corpus and Save

We now clean every document. For each one we:
1. Clean the `title` and `text`
2. Skip the document if it has **no usable text** after cleaning
3. Keep the original schema so downstream steps stay compatible

The output goes to `data/processed/02_clean_corpus.jsonl` — the input for the
chunking step (notebook 03).

In [9]:
# ── [9 / 10] Clean Every Document and Write Output ──────────────────────────

# Create the output folder if it does not already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

saved   = 0
skipped = 0

with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for doc in documents:
        clean_title = clean_text(doc.get("title", "") or "")
        clean_body  = clean_text(doc.get("text", "") or "")

        # A document with no body text is useless for retrieval — skip it
        if not clean_body:
            skipped += 1
            continue

        cleaned_doc = {
            "document_id"    : doc["document_id"],
            "title"          : clean_title,
            "text"           : clean_body,
            "source"         : doc.get("source", ""),
            "dataset_config" : doc.get("dataset_config", ""),
        }
        # ensure_ascii=False keeps real unicode characters readable in the file
        f.write(json.dumps(cleaned_doc, ensure_ascii=False) + "\n")
        saved += 1

print(f"Saved   : {saved:,} documents")
print(f"Skipped : {skipped} documents (empty after cleaning)")
print(f"Output  : {OUTPUT_FILE.resolve()}")

Saved   : 5,183 documents
Skipped : 0 documents (empty after cleaning)
Output  : /app/data/processed/02_clean_corpus.jsonl


---
## 10. Verify the Output File

We read the cleaned file back and re-run the exact problem scan from Step 4.
A successful cleaning run should report **zero** problems on every metric.

In [10]:
# ── [10 / 10] Verify the Output File ────────────────────────────────────────

clean_docs = []
with OUTPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        clean_docs.append(json.loads(line))

print(f"Lines in output file : {len(clean_docs):,}")
print()

# Re-run the same checks from Step 4 — every count should now be 0
remaining = {
    "edge_whitespace" : 0,
    "multi_space"     : 0,
    "multi_blank"     : 0,
    "control_chars"   : 0,
    "non_nfc_unicode" : 0,
}
for doc in clean_docs:
    text = doc["text"]
    if has_edge_whitespace(text):
        remaining["edge_whitespace"] += 1
    if re_multi_space.search(text):
        remaining["multi_space"] += 1
    if re_multi_blank.search(text):
        remaining["multi_blank"] += 1
    if re_control.search(text):
        remaining["control_chars"] += 1
    if text != unicodedata.normalize("NFC", text):
        remaining["non_nfc_unicode"] += 1

print("Remaining problems after cleaning:")
for name, n in remaining.items():
    print(f"  {name:<18}: {n:,}")
print()

if all(v == 0 for v in remaining.values()):
    print("Cleaning verified — no remaining text problems.")
else:
    print("WARNING: some problems remain — review the cleaning functions.")

print()
print("=" * 60)
print("First 2 cleaned documents")
print("=" * 60)
for i, doc in enumerate(clean_docs[:2]):
    print(f"\n--- Document {i} ---")
    for key, value in doc.items():
        display = str(value)[:100] + "..." if len(str(value)) > 100 else value
        print(f"  {key:<16} : {display}")

Lines in output file : 5,183

Remaining problems after cleaning:
  edge_whitespace   : 0
  multi_space       : 0
  multi_blank       : 0
  control_chars     : 0
  non_nfc_unicode   : 0

Cleaning verified — no remaining text problems.

First 2 cleaned documents

--- Document 0 ---
  document_id      : 4983
  title            : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion ten...
  text             : Alterations of the architecture of cerebral white matter in the developing human brain can affect co...
  source           : BeIR/scifact
  dataset_config   : corpus

--- Document 1 ---
  document_id      : 5836
  title            : Induction of myelodysplasia by myeloid-derived suppressor cells.
  text             : Myelodysplastic syndromes (MDS) are age-dependent stem cell malignancies that share biological featu...
  source           : BeIR/scifact
  dataset_config   : corpus
